# Notebook — Fonctionnalités

Outils annexes (dataset, sanity checks GT LARD, exports, vidéo).
Les phases generate / export sont dans `notebooks/generation.ipynb`.

> Note : certains helpers ciblent encore l'ancien layout (`runs/`, `footage/`) et doivent être adaptés au nouveau layout (`scenarios/`, `dataset/`, `images/`).

## 1. Setup

**À exécuter en premier.** 

In [ ]:
import sys
from pathlib import Path

def _find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "main.py").exists():
            return p
    return start

ROOT = _find_root(Path.cwd())

_project = str(ROOT / "sources")
if _project not in sys.path:
    sys.path.insert(0, _project)

from notebook_tools import (
    build_dataset, regroup_images,
    build_lard_box, show_sanity_lard,
    build_xplane_config, build_params_trace,
    build_esp,
    build_video,
    find_runs,  # adaptateur retro-compat -> find_scenarios
)
from scenario import SCENARIOS_DIR, DATASET_DIR

print(f"ROOT          = {ROOT}")
print(f"SCENARIOS_DIR = {SCENARIOS_DIR}  (exists={SCENARIOS_DIR.exists()})")
print(f"DATASET_DIR   = {DATASET_DIR}  (exists={DATASET_DIR.exists()})")

## 2. Lister les runs disponibles

Énumère tous les runs valides présents dans `runs/` (toutes générations
confondues), affichés en chemin relatif `<generation>/<run>`.

In [ ]:
runs = find_runs(all_runs=True)
for r in runs:
    print(f"  - {r.parent.name}/{r.name}")
print(f"\n{len(runs)} scenario(s) trouve(s).")

## 3. Création du dataset (arborescence par piste / scénario)

Format aligné sur le CSV LARD natif.

```
dataset/
├── metadata.csv                         # toutes pistes, tous scénarios
└── <ICAO_RWY>/                          # ex: KPDX_10L
    ├── metadata.csv                     # tous scénarios de cette piste
    └── <ICAO_RWY>_<NNN>/                # ex: KPDX_10L_001
        ├── metadata.csv                 # ce scénario uniquement
        └── images/000000.jpg ...
```


In [ ]:
build_dataset()

In [ ]:
# regroup_images() — regroupe + renumérote les images du dataset, avec metadata.csv.
#
# mode="piste" (defaut) : un dossier par piste
#     runs/dataset_regroup/<RWY>/img/000000.jpg ...
#     runs/dataset_regroup/<RWY>/metadata.csv      (scenarios de cette piste)
#
# mode="all"            : tout dans un seul dossier
#     runs/dataset_regroup/datasetr/img/000000.jpg ...
#     runs/dataset_regroup/datasetr/metadata.csv   (toutes pistes confondues)
#
# Les deux modes coexistent (chacun ne nettoie que son propre sous-dossier).

#regroup_images(mode="piste")
regroup_images(mode="all")

## 6. Générer `lard_box/` (images avec bbox GT LARD dessinées)

Usage :
- `build_lard_box()`                          : toutes les runs
- `build_lard_box("generation_01/KLAX_25R")`  : chemin composé si conflit

In [ ]:
# build_lard_box("generation_01/KLAX_25R")  # un seul run (chemin composé)
build_lard_box()

## 7. Sanity check : 3 images avec bbox GT LARD 

Affiche première / milieu / dernière image d'UN run avec les 4 coins GT LARD
(`degraded/` prio sinon `footage/`). Pas de mode "tous les runs" : sans argument,
prend le premier run trouvé.

Usage :
- `show_sanity_lard()`                          : premier run trouvé
- `show_sanity_lard("KLAX_25R")`                : un run cible (si unique)
- `show_sanity_lard("generation_01/KLAX_25R")`  : chemin composé si conflit

In [ ]:
# show_sanity_lard("generation_01/KLAX_25R")  # cible un run spécifique
show_sanity_lard()

## 8. Générer `xplane_config.json`


**Contenu** : `width`/`height` (résolution), `fov_h`/`fov_v` (champ de vision),
`pilot_eye_x/y/z` (position œil pilote sur X-Plane12 dépendant de l'avion choisi), `weather_status` (ok/absent).

Usage :
- `build_xplane_config()`                          : toutes les runs
- `build_xplane_config("generation_01/KLAX_25R")`  : chemin composé si conflit

In [ ]:
# build_xplane_config("generation_01/KLAX_25R")  # un seul run (chemin composé)
build_xplane_config()

## 9. Générer `params_trace.xml`

**Contenu** : 
 `trajectory` (fps + paramètres de trajectoire), `weather` (paramètres météo), `faults`
(fautes capteur appliquées).

Usage :
- `build_params_trace()`                          : toutes les runs
- `build_params_trace("generation_01/KLAX_25R")`  : chemin composé si conflit

In [ ]:
# build_params_trace("generation_01/KLAX_25R")  # un seul run (chemin composé)
build_params_trace()

## 10. Generer `.esp` (projet Google Earth Studio)

Reconstruit un projet GES (`<run>.esp`) depuis `poses_cam_export.json`
via LARD `GEODataset.create_scenario`. A importer dans Google Earth Studio
pour le rendu. `fov_vertical=30` = convention historique LARD-GES.

In [ ]:
# build_esp("generation_01/KLAX_25R")  # un seul run 
build_esp()


## 11. Générer une vidéo MP4 d'un run/des runs

Concatène les images du run (`degraded/` en priorité sinon `footage/`) en un MP4 (fps du yaml).

Usage :
- `build_video()`                          : toutes les runs
- `build_video(source="footage")`          : force la source (`footage` ou `degraded`)

In [ ]:
# build_video("generation_02/LFPO_25", source="footage")  # run spécifique + source footage
build_video() 